**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# CUDA in C++

> ⚠️ **Draft — code not machine-verified.** Requires an NVIDIA GPU + the CUDA toolkit (`nvcc`) not available at authoring time. An instructor should run each block before teaching. Remove this banner after that pass.

For students who outgrow Numba: raw CUDA C++ — explicit memory management, kernel launches, error handling, and the library ecosystem (cuBLAS/cuFFT) that usually beats hand-written kernels. Compile everything with `nvcc file.cu -o prog`.

## 1. Pre-requisites

[Intro to C](../Intro_Programming/Intro_C.ipynb) (pointers, malloc discipline); [Hardware-Accelerated Computing](./HW_Accelerated_Computing.ipynb) for the concepts.

---
### 🕐 Session 1 of 3 — *First Kernels* (~40 min)
**Goal:** the full lifecycle: allocate, copy, launch, synchronize, check errors, free.
**Feeds into:** Session 2 (memory management).

---

💡 **Intuition.** CUDA C++ is [Intro to C's](../Intro_Programming/Intro_C.ipynb) memory discipline with *two* heaps: host pointers and device pointers are the same type (`float*`) but live in different worlds, and mixing them up compiles fine then crashes at runtime. The `CUDA_CHECK` macro habit below is not optional style — kernel launches fail *silently* without it.

```cpp
// saxpy.cu — the "hello world" of CUDA
#include <cstdio>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do { \
    cudaError_t e = (call); \
    if (e != cudaSuccess) { \
        fprintf(stderr, "CUDA error %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(e)); \
        exit(1); } } while (0)

__global__ void saxpy(int n, float a, const float* x, float* y) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;   // same arithmetic as Numba's cuda.grid(1)
    if (i < n) y[i] = a * x[i] + y[i];
}

int main() {
    const int N = 1 << 24;
    float *x_h = (float*)malloc(N * sizeof(float));
    float *y_h = (float*)malloc(N * sizeof(float));
    for (int i = 0; i < N; ++i) { x_h[i] = 1.0f; y_h[i] = 2.0f; }

    float *x_d, *y_d;
    CUDA_CHECK(cudaMalloc(&x_d, N * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&y_d, N * sizeof(float)));
    CUDA_CHECK(cudaMemcpy(x_d, x_h, N * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(y_d, y_h, N * sizeof(float), cudaMemcpyHostToDevice));

    saxpy<<<(N + 255) / 256, 256>>>(N, 3.0f, x_d, y_d);
    CUDA_CHECK(cudaGetLastError());                  // catches bad launch configs
    CUDA_CHECK(cudaDeviceSynchronize());             // kernels are ASYNC: wait before trusting results

    CUDA_CHECK(cudaMemcpy(y_h, y_d, N * sizeof(float), cudaMemcpyDeviceToHost));
    printf("y[0] = %f (expect 5.0)\n", y_h[0]);
    cudaFree(x_d); cudaFree(y_d); free(x_h); free(y_h);
}
```

Build & run: `nvcc -O3 saxpy.cu -o saxpy && ./saxpy`

---
### 🕐 Session 2 of 3 — *Memory Management Patterns* (~35 min)
**Goal:** unified vs explicit memory, pinned transfers, and measuring bandwidth honestly.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (libraries).

---

**The three memory styles**, in ascending control:

```cpp
// 1. Unified memory — one pointer works on both sides; the driver migrates pages on demand
float* u; cudaMallocManaged(&u, N * sizeof(float));
// simplest code, unpredictable performance: great for prototyping, profile before shipping

// 2. Explicit device memory + regular host memory (Session 1) — the default

// 3. Pinned (page-locked) host memory — DMA-able, ~2x faster transfers, enables async overlap
float* p; cudaMallocHost(&p, N * sizeof(float));
cudaMemcpyAsync(x_d, p, N * sizeof(float), cudaMemcpyHostToDevice, stream);
```

Benchmark ritual (do this on your machine and keep the numbers):

```cpp
cudaEvent_t t0, t1; cudaEventCreate(&t0); cudaEventCreate(&t1);
cudaEventRecord(t0);
cudaMemcpy(x_d, x_h, N * sizeof(float), cudaMemcpyHostToDevice);
cudaEventRecord(t1); cudaEventSynchronize(t1);
float ms; cudaEventElapsedTime(&ms, t0, t1);
printf("H2D: %.1f GB/s\n", N * sizeof(float) / ms / 1e6);
// pageable vs pinned typically ~6 vs ~12 GB/s on PCIe 4 — measure yours
```

---
### 🕐 Session 3 of 3 — *The Library Ecosystem* (~35 min)
**Goal:** stop writing kernels: cuBLAS matmul and cuFFT spectra, correctly linked and checked.
**Builds on:** Session 2.

---

💡 **Intuition.** NVIDIA's library engineers have spent two decades on tiled matmuls; your [tiled kernel](./HW_Accelerated_Computing.ipynb) exists so you *understand* theirs. Production rule: kernel-write only what cuBLAS/cuFFT/cuDNN/Thrust don't already do.

```cpp
// gemm.cu — C = A·B via cuBLAS (column-major! the eternal gotcha)
#include <cublas_v2.h>
cublasHandle_t h; cublasCreate(&h);
const float one = 1.0f, zero = 0.0f;
// note the trick: computing B^T·A^T in column-major = A·B in row-major
cublasSgemm(h, CUBLAS_OP_N, CUBLAS_OP_N, n, m, k, &one, B_d, n, A_d, k, &zero, C_d, n);
cublasDestroy(h);
// link: nvcc gemm.cu -lcublas
```

```cpp
// spectrum.cu — cuFFT: the FFT from [Foundations 1 S7], at GB/s
#include <cufft.h>
cufftHandle plan;
cufftPlan1d(&plan, N, CUFFT_R2C, 1 /*batch*/);
cufftExecR2C(plan, signal_d, spectrum_d);        // in-place batched variants exist
cufftDestroy(plan);
// link: nvcc spectrum.cu -lcufft
```

Homework with teeth: benchmark your Session-1 saxpy, your tiled matmul, and cuBLAS on the
same sizes; plot GFLOP/s. The gap between your kernel and cuBLAS is the syllabus of a
graduate course — and the reason the library exists.

## 4. Conclusion

Explicit two-heap memory discipline, error-checked async launches, pinned transfers, and libraries first. You can now read any CUDA codebase — and know which parts you shouldn't write yourself.

---
## Where next

- [Hardware-Accelerated Computing](./HW_Accelerated_Computing.ipynb) — the performance levers, in Python where iteration is fast.
- [Intro to C](../Intro_Programming/Intro_C.ipynb) — the pointer discipline this workshop doubles down on.